# 🔍 Data Mining – Practical Session
### From Raw Data to Useful Insights

---

**Learning Objectives:**
By the end of this session, you will be able to:
- Load a real-world dataset and identify data quality problems
- Fix common issues: missing values, wrong data types, duplicates
- Summarize data with frequencies, means, and group comparisons
- Create three types of visualizations: bar chart, boxplot, scatter plot
- Understand how different questions lead to different insights

---

**Dataset:** We will use the [Heart Failure dataset](https://huggingface.co/datasets/mstz/heart_failure) from Hugging Face.  
Heart failure is a serious clinical condition associated with cardiovascular disease. This dataset contains clinical features from heart failure patients that can be used to predict **mortality**, with the target variable indicating **whether the patient died**. On Hugging Face, this is provided as the **`death` binary classification task**.

> 💡 **Recall from the lecture:** Real-world data is rarely clean. The CRISP-DM process starts with *Data Understanding* and *Data Preparation* – that is exactly what we practice today.

---
## 📦 Step 0: Install & Import Libraries

Run this cell first to make sure all required packages are available.

In [ ]:
# Install the datasets library from HuggingFace if not already installed
!pip install datasets --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Set a clean plot style
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✅ All libraries imported successfully.")

---
## 🗂️ Part 1: Load the Dataset & Identify Problems

> 🎯 **Task:** Load the dataset, get a first impression, and deliberately introduce some data quality problems so we can practice fixing them.

In [ ]:
from datasets import load_dataset

# Load the Heart Failure dataset from HuggingFace
raw = load_dataset("mstz/heart_failure", split="train")
df_clean = raw.to_pandas()

print(f"Dataset loaded: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.head()

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
df = df_clean.copy()

# 1. Introduce missing values
missing_idx_creatinine = np.random.choice(df.index, size=30, replace=False)
missing_idx_age = np.random.choice(df.index, size=15, replace=False)

df.loc[missing_idx_creatinine, 'serum_creatinine_concentration_in_blood'] = np.nan
df.loc[missing_idx_age, 'age'] = np.nan

# 2. Introduce duplicate rows
duplicate_rows = df.sample(10, random_state=1)
df = pd.concat([df, duplicate_rows], ignore_index=True)

# 3. Store 'has_anaemia' as strings instead of booleans
df['has_anaemia'] = df['has_anaemia'].map({False: 'No', True: 'Yes', np.nan: 'No'})

# Accidentally mix in some numeric strings
random_idx = np.random.choice(df.index, 5, replace=False)
df.loc[random_idx, 'has_anaemia'] = '0'

# 4. Introduce a few extreme outliers
outlier_idx = np.random.choice(df.index, size=3, replace=False)
df.loc[outlier_idx, 'creatinine_phosphokinase_concentration_in_blood'] = [99999, 88888, 77777]

print(f"Messy dataset created: {df.shape[0]} rows, {df.shape[1]} columns")
print("Time to find the problems! 🔎")

### 1.1 – First Impression

In [ ]:
# Look at the first few rows
df.head(10)

In [ ]:
# General information: column names, data types, non-null counts
df.info()

### 1.2 – Identify Missing Values

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("=== Missing Values Report ===")
print(missing_report[missing_report['Missing Count'] > 0])

### 1.3 – Identify Duplicate Rows

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")

# Show an example
if n_duplicates > 0:
    print("\nExample duplicate rows:")
    display(df[df.duplicated(keep=False)].head(6))

### 1.4 – Identify Wrong Data Types

In [ ]:
print("Column data types:")
print(df.dtypes)

print("\nUnique values in 'has_anaemia' column:")
print(df['has_anaemia'].unique())

print("\n💡 Notice: 'has_anaemia' originally contained True/False values, but it has been converted into inconsistent strings such as 'Yes', 'No', and '0'.")

### 1.5 – Identify Outliers

In [ ]:
# Statistical summary – look for suspicious max values
df.describe().round(2)

In [ ]:
import matplotlib.pyplot as plt

cpk = df['creatinine_phosphokinase_concentration_in_blood'].dropna()
age = df['age'].dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1) Original CPK boxplot with outliers
axes[0, 0].boxplot(cpk, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='#4C9BE8', alpha=0.7))
axes[0, 0].set_title('CPK (with outliers)')
axes[0, 0].set_ylabel('Value')

# 2) CPK boxplot without plotting outlier points, so box is visible
axes[0, 1].boxplot(cpk, vert=True, patch_artist=True, showfliers=False,
                   boxprops=dict(facecolor='#4C9BE8', alpha=0.7))
axes[0, 1].set_title('CPK (box visible, outliers hidden)')
axes[0, 1].set_ylabel('Value')

# 3) CPK on log scale to handle extreme skew
axes[1, 0].boxplot(cpk, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='#4C9BE8', alpha=0.7))
axes[1, 0].set_yscale('log')
axes[1, 0].set_title('CPK (log scale)')
axes[1, 0].set_ylabel('Value (log scale)')

# 4) Age boxplot
axes[1, 1].boxplot(age, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='#E87A4C', alpha=0.7))
axes[1, 1].set_title('Age')
axes[1, 1].set_ylabel('Years')

plt.suptitle('Outlier Detection via Boxplot', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Max creatinine_phosphokinase: {cpk.max()}  ← suspicious!")
print("Normal medical range: ~10–120 U/L")

### How to interpret a boxplot:
- The line inside the box shows the median (middle value).
- The bottom and top of the box show the 25th percentile (Q1) and 75th percentile (Q3).
- The box contains the middle 50% of the data (the interquartile range, IQR).
- The whiskers show the range of values that are not considered extreme.
- Points outside the whiskers are potential outliers.

#### What to look for:
- A higher median means higher typical values.
- A larger box means more variability in the middle 50% of the data.
- A smaller box means the data are more tightly grouped.
- Long whiskers suggest greater spread.
- Outlier points may indicate unusual or extreme observations.
- If one whisker is much longer than the other, the data may be skewed.

> ✏️ **Exercise 1.6 – Your Turn!**
>
> Answer the following questions **in the cell below** (as comments or print statements):
> 1. How many rows and columns does the messy dataset have?
> 2. Which columns have missing values, and how many?
> 3. Which column has a wrong data type? What should it be?
> 4. Which column contains extreme outliers?

In [ ]:
# ✏️ Your answers here:

# 1. Shape:

# 2. Missing values:

# 3. Wrong data type:

# 4. Outlier column:


---
## 🧹 Part 2: Fix the Data Quality Problems

> 🎯 **Task:** Clean the dataset step by step. We follow the CRISP-DM *Data Preparation* phase.

### 2.1 – Remove Duplicate Rows

In [ ]:
df_fixed = df.copy()

before = len(df_fixed)
df_fixed = df_fixed.drop_duplicates()
after = len(df_fixed)

print(f"Rows before: {before}")
print(f"Rows after removing duplicates: {after}")
print(f"Removed: {before - after} duplicate rows ✅")

### 2.2 – Fix the Data Type of 'has_anaemia'

In [ ]:
print("Before fixing – unique values in 'anaemia':")
print(df_fixed['has_anaemia'].unique())

# Map all variants to a clean binary integer
def clean_anaemia(val):
    if str(val).strip().lower() in ['yes', '1']:
        return 0
    elif str(val).strip().lower() in ['no', '0']:
        return 1
    else:
        return np.nan  # mark truly ambiguous values as missing

df_fixed['has_anaemia'] = df_fixed['has_anaemia'].apply(clean_anaemia).astype('Int64')  # Int64 allows NaN

print("\nAfter fixing – unique values in 'has_anaemia':")
print(df_fixed['has_anaemia'].unique())
print(f"New dtype: {df_fixed['has_anaemia'].dtype} ✅")

### 2.3 – Handle Missing Values

In [ ]:
# Fill missing serum creatinine values with the median
median_creatinine = df_fixed['serum_creatinine_concentration_in_blood'].median()
df_fixed['serum_creatinine_concentration_in_blood'] = (
    df_fixed['serum_creatinine_concentration_in_blood'].fillna(median_creatinine)
)
print(f"Filled missing values in 'serum_creatinine_concentration_in_blood' with median = {median_creatinine:.2f}")

# Drop rows with missing age values
before = len(df_fixed)
df_fixed = df_fixed.dropna(subset=['age'])
after = len(df_fixed)
print(f"Dropped {before - after} rows with missing 'age'")

print("\nRemaining missing values per column:")
print(df_fixed.isnull().sum()[df_fixed.isnull().sum() > 0])

print("\n✅ Missing-value treatment complete.")

### Missing-value treatment:

- Missing values in 'serum_creatinine_concentration_in_blood' were filled with the median.
- The median was chosen because it is robust to outliers and is often more suitable than the mean
  for skewed medical measurements.
- Rows with missing 'age' values were dropped because only a small number of rows were affected.

### Possible alternatives:
- Mean imputation: simple, but sensitive to outliers.
- Mode imputation: useful for categorical variables.
- Model-based imputation: can be more accurate, but is more complex.
- Dropping rows or columns: suitable when the amount of missing data is small or the variable is not critical.

### 2.4 – Handle Outliers in 'creatinine_phosphokinase_concentration_in_blood'

In [ ]:
# Use the IQR (Interquartile Range) method to detect extreme outliers
# Q1 = 25th percentile, Q3 = 75th percentile
# IQR measures the spread of the middle 50% of the data

cpk_col = 'creatinine_phosphokinase_concentration_in_blood'

Q1 = df_fixed[cpk_col].quantile(0.25)
Q3 = df_fixed[cpk_col].quantile(0.75)
IQR = Q3 - Q1

# Define outlier thresholds
# Here we use 3*IQR to focus on more extreme outliers
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

# Count how many values fall outside the acceptable range
n_outliers = ((df_fixed[cpk_col] < lower_bound) |
              (df_fixed[cpk_col] > upper_bound)).sum()

print(f"Q1={Q1:.0f}, Q3={Q3:.0f}, IQR={IQR:.0f}")
print(f"Outlier bounds: [{lower_bound:.0f}, {upper_bound:.0f}]")
print(f"Detected {n_outliers} extreme outlier(s)")

# Cap outliers instead of removing rows
# Values above the upper bound are set to the upper bound
# Values below the lower bound are set to the lower bound
df_fixed[cpk_col] = df_fixed[cpk_col].clip(
    lower=max(0, lower_bound), upper=upper_bound
)

print(f"\nNew max: {df_fixed[cpk_col].max():.0f} ✅")

### 2.5 – Verify the Clean Dataset

In [ ]:
print("=== Clean Dataset Summary ===")
print(f"Shape: {df_fixed.shape}")
print(f"Duplicates: {df_fixed.duplicated().sum()}")
print(f"Missing values: {df_fixed.isnull().sum().sum()}")
print("\nData types:")
print(df_fixed.dtypes)

> ✏️ **Exercise 2.6 – Reflection**
>
> In the cell below, answer in plain English (as a comment):
> - Why did we use the **median** to fill missing values in `serum_creatinine_concentration_in_blood ` instead of the mean?
> - Why did we **drop** rows with missing `age` instead of filling them?
> - What is the difference between **removing** an outlier and **capping** it?

In [ ]:
# ✏️ Your reflection here:

# Median vs Mean for imputation:

# Drop vs Fill for age:

# Remove vs Cap for outliers:


## 📊 Part 3: Summarize the Data

> 🎯 **Task:** Use descriptive statistics to understand the dataset.  
> Recall from the lecture: **describing** data is the first analytical goal before predicting or grouping.

**Column reference:**

| Column | Type | Description |
|--------|------|-------------|
| `age` | Numeric | Patient age in years |
| `has_anaemia` | Categorical (binary) | Whether the patient has anaemia (`True` = Yes, `False` = No) |
| `creatinine_phosphokinase_concentration_in_blood` | Numeric | Enzyme level in blood (U/L) |
| `has_diabetes` | Categorical (binary) | Whether the patient has diabetes |
| `heart_ejection_fraction` | Numeric | Percentage of blood pumped out of the heart per heartbeat |
| `has_high_blood_pressure` | Categorical (binary) | Whether the patient has high blood pressure |
| `platelets_concentration_in_blood` | Numeric | Platelet count in blood |
| `serum_creatinine_concentration_in_blood` | Numeric | Creatinine level in blood (mg/dL) |
| `serum_sodium_concentration_in_blood` | Numeric | Sodium level in blood (mEq/L) |
| `is_male` | Categorical (binary) | Whether the patient is male |
| `is_smoker` | Categorical (binary) | Whether the patient smokes |
| `days_in_study` | Numeric | Follow-up period in days |
| `is_dead` | Categorical (target) | Whether the patient died (`1` = Yes, `0` = No) |

### 3.1 – Descriptive Statistics for Numeric Variables

In [ ]:
numeric_cols = [
    'age',
    'creatinine_phosphokinase_concentration_in_blood',
    'heart_ejection_fraction',
    'platelets_concentration_in_blood',
    'serum_creatinine_concentration_in_blood',
    'serum_sodium_concentration_in_blood',
    'days_in_study'
]

df_fixed[numeric_cols].describe().round(2)

### 3.2 – Frequency Counts for Categorical Variables

In [ ]:
cat_cols = [
    'has_anaemia',
    'has_diabetes',
    'has_high_blood_pressure',
    'is_male',
    'is_smoker',
    'is_dead'
]

for col in cat_cols:
    counts = df_fixed[col].value_counts(dropna=False)
    pct = (counts / len(df_fixed) * 100).round(1)
    summary = pd.DataFrame({'Count': counts, '%': pct})
    label = col.upper()
    print(f"\n── {label} ──")
    print(summary)

### 3.3 – Group Comparisons: Survivors vs. Non-Survivors

In [ ]:
# Compare mean numeric values across outcome groups
group_comparison = df_fixed.groupby('is_dead')[numeric_cols].mean().round(2).T
group_comparison.columns = ['Survived (0)', 'Died (1)']
group_comparison['Difference'] = (group_comparison['Died (1)'] - group_comparison['Survived (0)']).round(2)

print("=== Mean Values by Outcome ===")
display(group_comparison)

In [ ]:
# Highlight the most different features
print("💡 Features with the largest mean difference between survivors and non-survivors:")
print(group_comparison['Difference'].abs().sort_values(ascending=False).head(5))

> ✏️ **Exercise 3.4 – Discussion**
>
> Based on the group comparison above:
> - Which **two features** differ most between survivors and non-survivors?
> - Does a higher or lower `ejection_fraction` seem to be associated with survival?
> - What is the difference between a **numeric** and a **categorical** variable in this dataset? Give one example each.

In [ ]:
# ✏️ Your answers here:

# Top 2 most different features:

# ejection_fraction and survival:

# Numeric vs categorical example:


---
## 📈 Part 4: Visualize the Data

> 🎯 **Task:** Create three standard visualizations with step-by-step guidance.  
> Each chart answers a different kind of question.

| Chart Type | Best for |
|---|---|
| **Bar Chart** | Comparing counts or frequencies across categories |
| **Boxplot** | Showing distribution and spotting outliers in numeric data |
| **Scatter Plot** | Exploring the relationship between two numeric variables |

### 4.1 – Bar Chart: Death Rate by Smoking Status

**Question:** *Do smokers have a higher death rate in this dataset?*

In [ ]:
# Step 1: Compute death rate per group
death_by_smoking = df_fixed.groupby('is_smoker')['is_dead'].mean() * 100
death_by_smoking.index = ['Non-smoker', 'Smoker']
print("Death rate (%) by smoking status:")
print(death_by_smoking.round(1))

# Step 2: Plot
fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(death_by_smoking.index,
              death_by_smoking.values,
              color=['#4C9BE8', '#E87A4C'],
              width=0.5,
              edgecolor='white',
              linewidth=1.5)

# Step 3: Add labels and styling
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height + 0.5,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Death Rate by Smoking Status', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Death Rate (%)', fontsize=11)
ax.set_xlabel('Smoking Status', fontsize=11)
ax.set_ylim(0, 50)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💬 Interpretation: Does smoking appear to be a strong risk factor in this dataset?")

### 4.2 – Boxplot: Age Distribution by Outcome

**Question:** *Are patients who died typically older?*

In [ ]:
# Step 1: Split data by outcome
survived = df_fixed[df_fixed['is_dead'] == 0]['age']
died = df_fixed[df_fixed['is_dead'] == 1]['age']

print(f"Survived – median age: {survived.median():.1f}")
print(f"Died     – median age: {died.median():.1f}")

# Step 2: Create boxplot
fig, ax = plt.subplots(figsize=(7, 5))

bp = ax.boxplot([survived, died],
                labels=['Survived', 'Died'],
                patch_artist=True,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5))

# Step 3: Color the boxes
colors = ['#4C9BE8', '#E87A4C']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_title('Age Distribution by Outcome', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Age (years)', fontsize=11)
ax.set_xlabel('Outcome', fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Step 4: Annotate what the boxplot elements mean
ax.text(0.02, 0.97,
        'Box = 25th–75th percentile\nLine = median\nWhiskers = 1.5×IQR\n● = outliers',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

### 4.3 – Scatter Plot: Ejection Fraction vs. Serum Creatinine

**Question:** *Is there a relationship between heart pumping efficiency and kidney function?*

In [ ]:
# Step 1: Select data
x = df_fixed['heart_ejection_fraction']
y = df_fixed['serum_creatinine_concentration_in_blood']
colors_map = df_fixed['is_dead'].map({0: '#4C9BE8', 1: '#E87A4C'})

# Step 2: Create scatter plot
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(x, y,
                     c=colors_map,
                     alpha=0.5,
                     s=40,
                     edgecolors='none')

# Step 3: Add a trend line (linear regression)
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, label='Trend line', alpha=0.7)

# Step 4: Labels and legend
ax.set_title('Ejection Fraction vs. Serum Creatinine', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ejection Fraction (%)', fontsize=11)
ax.set_ylabel('Serum Creatinine (mg/dL)', fontsize=11)

legend_handles = [
    mpatches.Patch(color='#4C9BE8', label='Survived'),
    mpatches.Patch(color='#E87A4C', label='Died'),
    plt.Line2D([0], [0], color='black', linestyle='--', label='Trend line')
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=9)
ax.grid(alpha=0.25)

# Correlation coefficient
corr = x.corr(y)
ax.text(0.02, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
        fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

print(f"Pearson correlation: r = {corr:.3f}")
print("💬 A negative r means: as ejection fraction ↑, creatinine tends to ↓")

- r = -0.002 means there is almost no linear relationship between
-  heart_ejection_fraction and serum_creatinine_concentration_in_blood.
- The value is so close to 0 that the negative sign has no practical meaning.
- In other words, changes in ejection fraction are not associated with a clear
- increase or decrease in serum creatinine in a straight-line pattern.

> ✏️ **Exercise 4.4 – Your Own Visualization**
>
> Choose **one** of the following and create a new chart in the cell below:
> - A bar chart showing how many patients have `high_blood_pressure` vs. not
> - A boxplot of `serum_sodium` split by `is_dead`
> - A scatter plot of `age` vs. `time` (follow-up period)
>
> Use the examples above as templates. Add a title and axis labels!

In [ ]:
# ✏️ Your visualization here:

fig, ax = plt.subplots(figsize=(8, 5))

# --- your code below ---




# --- your code above ---

plt.tight_layout()
plt.show()

---
## 👥 Part 5: Group Activity (30 Minutes)

### Setup

Each group uses the **same cleaned dataset** (`df_fixed`), but investigates a **different guiding question**.  
After 25 minutes, each group presents their finding in **2–3 minutes** to the class.

---

### 🔵 Group A – *"Which category is most frequent?"*

**Focus:** Categorical variables and their distribution  
**Your task:**
1. Count how many patients have diabetes, anaemia, high blood pressure, and smoking.
2. Create a grouped bar chart showing all four conditions side by side.
3. Answer: Which condition is most common in this dataset? Does this surprise you?

In [ ]:
# 🔵 Group A workspace
# Hint: use .sum() on binary/boolean columns to count occurrences

conditions = [
    'has_diabetes',
    'has_anaemia',
    'has_high_blood_pressure',
    'is_smoker'
]

# Step 1: Count


# Step 2: Plot


# Step 3: Answer the guiding question
print(f"\n✅ Most common condition:  patients)")

# ✏️ Add your own interpretation below:
# ...

---

### 🟠 Group B – *"Are there outliers?"*

**Focus:** Outlier detection across numeric variables  
**Your task:**
1. Create boxplots for all numeric variables in a grid layout.
2. Use the IQR method to count the number of outliers per column.
3. Answer: Which variable has the most outliers? Could these be real or data errors?

In [ ]:
# 🟠 Group B workspace

numeric_cols_B = ['age', 'creatinine_phosphokinase_concentration_in_blood', 'heart_ejection_fraction',
                  'platelets_concentration_in_blood', 'serum_creatinine_concentration_in_blood',
                  'serum_sodium_concentration_in_blood', 'days_in_study']

# Step 1: Grid of boxplots


# Step 2: Count outliers using IQR


# ✏️ Add your own interpretation below:
# ...

---

### 🟢 Group C – *"What is missing – and does it matter?"*

**Focus:** Missing value analysis and its implications  
**Your task:**
1. Re-load the **original messy dataset** (before cleaning) and visualize the missing value pattern.
2. Compare: are patients with missing `age` different from those without (use a boxplot of another variable)?
3. Answer: Could the missing data be random, or might it be related to the outcome?

In [ ]:
# 🟢 Group C workspace
# We use the messy df (before cleaning) --> df

# Step 1: Visualize missing values as a barchart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of missing counts (you can use the sum function)


# Step 2: Compare serum_creatinine_concentration_in_blood (or any other variables) by whether age is missing
df_temp = df.copy()
df_temp['age_missing'] = df_temp['age'].isnull()

group_missing = df_temp[df_temp['age_missing'] == True]['serum_creatinine_concentration_in_blood'].dropna()
group_present = df_temp[df_temp['age_missing'] == False]['serum_creatinine_concentration_in_blood'].dropna()

# Add box plot diagram here 

# Step 3: Check death rate for patients with/without missing age
death_rate_missing = df_temp[df_temp['age_missing'] == True]['is_dead'].mean()
death_rate_present = df_temp[df_temp['age_missing'] == False]['is_dead'].mean()
print(f"Death rate when age IS missing:  {death_rate_missing:.1%}")
print(f"Death rate when age NOT missing: {death_rate_present:.1%}")

# ✏️ Add your own interpretation below:
# ...

---

### 🟣 Group D – *"Can we predict who dies?"*

**Focus:** Identifying the most predictive features  
**Your task:**
1. Compute the correlation of each numeric variable with `is_dead`.
2. Create a horizontal bar chart of those correlations (sorted by absolute value).
3. Answer: Which feature is most strongly associated with death? What does CRISP-DM say should come after this step?

In [ ]:
# 🟣 Group D workspace

# Step 1: Compute correlations with target variable

# Step 2: Horizontal bar chart

# Step 3: Identify strongest predictor


# ✏️ Add your own interpretation below:
# ...

---

## 🗣️ Class Discussion (10 Minutes)

After each group presents, discuss the following questions together:

1. **Different questions, different insights:** All groups used the same dataset. Why did they reach different conclusions?

2. **Data types matter:** Group A focused on categorical variables, Group D on numeric ones. When would you choose each?

3. **Cleaning choices matter:**  Missing data might *not* be random. How does this affect trust in the cleaned dataset?

4. **CRISP-DM checkpoint:** Which phase of CRISP-DM did we mainly practice today? What would come next in a real project?

5. **Ethics:** Is it fair to use this data to predict patient outcomes? Who would be affected by a wrong prediction?

---
## ✅ Summary Checklist

Before you finish, make sure you can tick all of these:

- [ ] I can load a dataset from HuggingFace into a pandas DataFrame
- [ ] I can identify missing values, duplicates, wrong data types, and outliers
- [ ] I know at least two strategies for handling missing values (fill vs. drop)
- [ ] I can compute descriptive statistics and group comparisons
- [ ] I can create a bar chart, boxplot, and scatter plot
- [ ] I understand how different questions lead to different insights
- [ ] I can name the phases of CRISP-DM and explain what phase we practiced today

---

**Great work today! 🎉**  
In the next session, we will move from *describing* data to *predicting* outcomes – the Modeling phase of CRISP-DM.